# broadcasting-rules — faded example 1: Subtract each row's minimum via row broadcast

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcasting-rules`. Running the beacon reports progress on the `Numpy: Vectorization and broadcasting` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `broadcasting-rules`**, which bridges to the bank subtopic `Numpy: Vectorization and broadcasting` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

A reduction with `keepdim=True` preserves the reduced axis as size 1 so the result broadcasts straight back against the original tensor. Reducing `(N, D)` over the column axis with `keepdim=True` yields `(N, 1)`, which then column-broadcasts across the `D` features.

## Faded exercise 1

### Faded — subtract each row's minimum

Implement `row_minus_min(x)`: given `x` of shape `(N, D)`, subtract the **minimum of each row** from every element of that row. Output shape `(N, D)`; `out[n, d] == x[n, d] - min_over_d(x[n])`.

The scaffold computes the result by subtracting a per-row minimum. You must fill in the line that computes that per-row minimum so it broadcasts correctly back against `x`.

**Fill in:** the per-row minimum of x reduced over the feature axis, kept as shape (N, 1) so it column-broadcasts back across the D features

In [ ]:
def row_minus_min(x):
    row_min = ____  # TODO: the per-row minimum of x reduced over the feature axis, kept as shape (N, 1) so it column-broadcasts back across the D features
    return x - row_min


def _test():
    t.manual_seed(0)
    x = t.randn(5, 7)
    out = row_minus_min(x)
    assert out.shape == (5, 7), out.shape
    # Independent reference: every row's min must become exactly 0.
    ref = x - x.min(dim=1, keepdim=True).values
    assert t.allclose(out, ref, atol=1e-6)
    assert t.allclose(out.min(dim=1).values, t.zeros(5), atol=1e-6)
    assert (out >= -1e-6).all()


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def row_minus_min(x):
    row_min = x.min(dim=1, keepdim=True).values   # (N, 1)
    return x - row_min
```
</details>